<a href="https://colab.research.google.com/github/BotCalvin/BUS-118S/blob/main/Exercise_1.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [18]:
import json

# -----------------------------
# Example customer message (you can change this)
# -----------------------------
customer_message = "My order says delivered but I never received it. I need it by tomorrow."

# -----------------------------
# Prompt templates (what you'd paste into ChatGPT/LLM)
# NOTE: We are simulating LLM outputs below since you have no API key.
# -----------------------------

# V1 (INTENTIONALLY WEAKER): often fails because it doesn't strictly force schema details
PROMPT_STEP1_CLASSIFY_V1 = f"""
You are a customer support triage agent.

Task:
Classify the customer's message into a single category and urgency, and summarize the issue.

Constraints:
- Be concise and professional.
- Do NOT invent facts not in the message.
- Output MUST be valid JSON only (no extra text).
- JSON keys: category, urgency, summary, risk_flags
- category must be one of: "billing", "delivery", "technical", "account", "returns", "other"
- urgency must be one of: "low", "medium", "high"
- risk_flags must be an array; include any of: "fraud", "chargeback", "security", "legal", "safety", "none"

Customer message:
"{customer_message}"
""".strip()

# V2 (IMPROVED): stricter schema + word limit + explicit JSON-only instruction
PROMPT_STEP1_CLASSIFY_V2 = f"""
You are a customer support triage agent.

Task:
Classify the customer’s message into category + urgency, and produce a short summary.

Hard constraints (follow exactly):
1) Output MUST be VALID JSON ONLY. No markdown, no commentary.
2) Do NOT add facts not stated.
3) Use this exact JSON schema:
{{
  "category": "billing|delivery|technical|account|returns|other",
  "urgency": "low|medium|high",
  "summary": "string (<= 25 words)",
  "risk_flags": ["fraud|chargeback|security|legal|safety|none", ...]
}}
4) If no risk flags apply, set risk_flags to ["none"].

Customer message:
"{customer_message}"
""".strip()

# IMPORTANT: this uses .format(), so JSON braces must be escaped as {{ }}
PROMPT_STEP2_GATHER_INFO = """
You are a customer support agent. You have already triaged the issue.

Inputs:
- Customer message: "{customer_message}"
- Triage JSON: {triage_json}

Task:
Ask ONLY the minimum missing questions needed to resolve the issue for the given category.

Constraints:
- Ask at most 4 questions.
- Questions must be short and specific.
- Do NOT propose solutions yet.
- Output MUST be valid JSON only.
- JSON schema:
{{
  "questions": ["...", "..."]
}}
""".strip()

PROMPT_STEP3_PROPOSE_SOLUTION = """
You are a customer support agent.

Inputs:
- Customer message: "{customer_message}"
- Triage JSON: {triage_json}
- Missing-info questions JSON: {questions_json}

Task:
Provide a helpful next-step resolution plan based on the triage. If key info is missing, include conditional steps.

Constraints:
- Tone: calm, professional, empathetic.
- Do NOT blame the customer.
- Do NOT request sensitive info (full card number, SSN, passwords).
- Output MUST be valid JSON only.
- JSON schema:
{{
  "response_to_customer": "string",
  "internal_actions": ["...", "..."]
}}
""".strip()

PROMPT_STEP4_ESCALATION_RULE = """
You are a support operations policy checker.

Inputs:
- Triage JSON: {triage_json}
- Proposed solution JSON: {solution_json}

Task:
Decide whether to escalate to a human agent and why.

Escalate if ANY are true:
- urgency is "high"
- risk_flags contains anything other than "none"
- the issue involves a promised delivery deadline within 24 hours
- customer reports non-receipt after "delivered" status (possible carrier investigation)

Constraints:
- Output MUST be valid JSON only.
- JSON schema:
{{
  "escalate": true/false,
  "reason": "string",
  "queue": "string (e.g., delivery-specialist|billing-specialist|security|general)"
}}
""".strip()

print("Setup complete. Prompts loaded (with escaped JSON braces for .format()).")

Setup complete. Prompts loaded (with escaped JSON braces for .format()).


In [19]:
def parse_json_or_error(text):
    try:
        return json.loads(text), None
    except Exception as e:
        return None, str(e)

def show(title, text):
    print("\n" + "="*70)
    print(title)
    print("="*70)
    print(text)

# ----------------------------------------------------
# SIMULATED "LLM" OUTPUTS (since you don't have an API)
# You can replace these strings with real ChatGPT outputs if required.
# ----------------------------------------------------

# Step 1 - V1 output (INTENTIONALLY BAD: not JSON -> shows iteration/testing)
SIM_STEP1_OUTPUT_V1 = "Category: delivery, urgency: high. Summary: Package shows delivered but missing."

# Step 1 - V2 output (VALID JSON)
SIM_STEP1_OUTPUT_V2 = json.dumps({
    "category": "delivery",
    "urgency": "high",
    "summary": "Order marked delivered but customer did not receive it; needs it by tomorrow.",
    "risk_flags": ["none"]
})

# Step 2 output (VALID JSON)
SIM_STEP2_OUTPUT = json.dumps({
    "questions": [
        "What is the order number?",
        "What is the delivery address and any delivery instructions (e.g., gate code)?",
        "When did the tracking show 'delivered' (date/time)?",
        "Have you checked with neighbors or a front desk/mailroom?"
    ]
})

# Step 3 output (VALID JSON)
SIM_STEP3_OUTPUT = json.dumps({
    "response_to_customer": (
        "I’m sorry this happened. Since the package shows ‘delivered’ but you didn’t receive it, "
        "please share your order number and confirm the delivery address. Also tell me the delivered "
        "timestamp and whether you checked with neighbors/mailroom. If it can’t be located quickly, "
        "we can start a carrier investigation and arrange a replacement or refund depending on stock."
    ),
    "internal_actions": [
        "Verify tracking details and delivered scan location (if available).",
        "Check for carrier photo/proof-of-delivery.",
        "If within policy, open carrier investigation ticket.",
        "If customer needs item by tomorrow, check expedited replacement eligibility."
    ]
})

# Step 4 output (VALID JSON)
SIM_STEP4_OUTPUT = json.dumps({
    "escalate": True,
    "reason": "High urgency and 'delivered but not received' requires carrier investigation and time-sensitive handling.",
    "queue": "delivery-specialist"
})

print("Simulation helpers ready.")

Simulation helpers ready.


In [20]:
# -----------------------------
# STEP 1 — Classification (V1 -> fails) then iterate to V2
# -----------------------------
show("STEP 1 PROMPT (V1 - BEFORE)", PROMPT_STEP1_CLASSIFY_V1)
show("STEP 1 OUTPUT (V1 - BEFORE)", SIM_STEP1_OUTPUT_V1)

triage_obj, err = parse_json_or_error(SIM_STEP1_OUTPUT_V1)
if err:
    print("\n✅ Iteration evidence: V1 failed JSON parsing:", err)
    show("STEP 1 PROMPT (V2 - AFTER)", PROMPT_STEP1_CLASSIFY_V2)
    show("STEP 1 OUTPUT (V2 - AFTER)", SIM_STEP1_OUTPUT_V2)
    triage_obj, err = parse_json_or_error(SIM_STEP1_OUTPUT_V2)

assert err is None, "Step 1 V2 should be valid JSON"
triage_json = json.dumps(triage_obj)

# -----------------------------
# STEP 2 — Gather missing info (uses triage output)
# -----------------------------
prompt2 = PROMPT_STEP2_GATHER_INFO.format(
    customer_message=customer_message,
    triage_json=triage_json
)
show("STEP 2 PROMPT", prompt2)
show("STEP 2 OUTPUT", SIM_STEP2_OUTPUT)

questions_obj, err = parse_json_or_error(SIM_STEP2_OUTPUT)
assert err is None, "Step 2 should be valid JSON"
questions_json = json.dumps(questions_obj)

# -----------------------------
# STEP 3 — Propose solution (uses triage + questions)
# -----------------------------
prompt3 = PROMPT_STEP3_PROPOSE_SOLUTION.format(
    customer_message=customer_message,
    triage_json=triage_json,
    questions_json=questions_json
)
show("STEP 3 PROMPT", prompt3)
show("STEP 3 OUTPUT", SIM_STEP3_OUTPUT)

solution_obj, err = parse_json_or_error(SIM_STEP3_OUTPUT)
assert err is None, "Step 3 should be valid JSON"
solution_json = json.dumps(solution_obj)

# -----------------------------
# STEP 4 — Escalation decision (uses triage + solution)
# -----------------------------
prompt4 = PROMPT_STEP4_ESCALATION_RULE.format(
    triage_json=triage_json,
    solution_json=solution_json
)
show("STEP 4 PROMPT", prompt4)
show("STEP 4 OUTPUT", SIM_STEP4_OUTPUT)

escalation_obj, err = parse_json_or_error(SIM_STEP4_OUTPUT)
assert err is None, "Step 4 should be valid JSON"

print("\n🎉 Chain complete. Final outputs:")
print("Triage:", triage_obj)
print("Questions:", questions_obj)
print("Solution:", solution_obj)
print("Escalation:", escalation_obj)


STEP 1 PROMPT (V1 - BEFORE)
You are a customer support triage agent.

Task:
Classify the customer's message into a single category and urgency, and summarize the issue.

Constraints:
- Be concise and professional.
- Do NOT invent facts not in the message.
- Output MUST be valid JSON only (no extra text).
- JSON keys: category, urgency, summary, risk_flags
- category must be one of: "billing", "delivery", "technical", "account", "returns", "other"
- urgency must be one of: "low", "medium", "high"
- risk_flags must be an array; include any of: "fraud", "chargeback", "security", "legal", "safety", "none"

Customer message:
"My order says delivered but I never received it. I need it by tomorrow."

STEP 1 OUTPUT (V1 - BEFORE)
Category: delivery, urgency: high. Summary: Package shows delivered but missing.

✅ Iteration evidence: V1 failed JSON parsing: Expecting value: line 1 column 1 (char 0)

STEP 1 PROMPT (V2 - AFTER)
You are a customer support triage agent.

Task:
Classify the customer’